In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# Find the repo root and add it to sys.path FIRST
repo = Path.cwd()
while not (repo / "functions").exists() and repo.parent != repo:
    repo = repo.parent
sys.path[:0] = [str(repo), str(Path.cwd().parent.parent)]

import tempfile, py7zr
import geopandas as gpd
import glob, json
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from rasterio.warp import reproject, calculate_default_transform

from constants import *
from functions.plot_hillshade import plot_hillshade
from utils import *
from fp_plotting import *
from ecosystems.cs.cs_plotting import *
from ecosystems.cs.cs_data_process import *


WP_PATH = Path("/Users/kylabazlen/Documents/Climate_Roadmap/Forest/Data/watershed_protection/watershed_protection.tif")


### PLOT BASE WATERSHED PROTECTION LAYER LAYER ###

In [ ]:
import xml.etree.ElementTree as ET
from matplotlib.colors import LinearSegmentedColormap, Normalize

def parse_qml_symbology(qml_path):
    """Parse a QGIS .qml file and return a matplotlib cmap + norm + opacity."""
    tree = ET.parse(qml_path)
    root = tree.getroot()

    renderer = root.find(".//rasterrenderer")
    opacity = float(renderer.get("opacity", 1))

    shader = renderer.find(".//colorrampshader")
    vmin = float(shader.get("minimumValue"))
    vmax = float(shader.get("maximumValue"))

    # Collect (value, hex_color) stops in order
    stops = []
    for item in shader.findall("item"):
        value = float(item.get("value"))
        color = item.get("color")
        alpha = int(item.get("alpha", 255)) / 255
        stops.append((value, color, alpha))

    stops.sort(key=lambda s: s[0])

    # Normalize stop positions to 0..1 for the colormap
    span = vmax - vmin
    cmap_stops = [((v - vmin) / span, c) for v, c, _ in stops]
    cmap = LinearSegmentedColormap.from_list("from_qml", cmap_stops, N=256)

    norm = Normalize(vmin=vmin, vmax=vmax)
    return cmap, norm, opacity, stops

QML_PATH = "/Users/kylabazlen/Documents/Climate_Roadmap/Forest/Data/watershed_protection/Watershed_Protection.qml"
cmap, norm, opacity, stops = parse_qml_symbology(QML_PATH)

print(f"Opacity: {opacity}")
print(f"Color stops:")
for v, c, a in stops:
    print(f"  value={v}  color={c}  alpha={a}")

In [ ]:
wp_arr, wp_extent = reproject_raster(WP_PATH, TARGET_CRS)
wp_arr = np.ma.masked_where(
    (wp_arr == -3.4028234663852886e+38) | np.isnan(wp_arr),
    wp_arr,
)

# Coarsen for faster plotting
step = 2
wp_small = wp_arr[::step, ::step]

# Hillshade base
fig, ax = plot_hillshade()

# Watershed protection layer on top
im = ax.imshow(wp_small, extent=wp_extent, cmap=cmap, norm=norm,
               interpolation="nearest", alpha=0.75, zorder=2)
cbar = fig.colorbar(im, ax=ax, shrink=0.6, label="Watershed Protection (0–100)")
cbar.set_ticks([])

# Counties on top
plt_counties(COUNTIES_PATH, county_edgecolor="black", county_linewidth=1.0, ax=ax)

plt.tight_layout()
plt.show()

## Bivar plots ##

In [ ]:
# ============================================================
# Pipeline
# ============================================================

LAYERS_TO_PLOT = ["Days Exceeding 95th Percentile Maximum Temperature"] #["drought__D3"] #["5 day max precip"]
#, "Days Exceeding 95th Percentile Maximum Temperature"
TARGET_CRS = "EPSG:3857"
N_CLASSES = 3

CS_PATH = Path(
    "/Users/kylabazlen/Documents/Climate_Roadmap/Ecosystems/eco_data/"
    "COStatewideConservationSummaryV8/TIF_File/ConservationSummaryV8_NoTribalLands.tif"
)
OUT_ROOT = Path("/Users/kylabazlen/Documents/Climate_Roadmap/outputs/bivariate")

# CS = fine reference grid — build once
cs_transform, cs_width, cs_height = build_reference_grid(str(CS_PATH), dst_crs=TARGET_CRS)
cs_aligned = reproject_to_grid(
    str(CS_PATH), cs_transform, cs_width, cs_height,
    dst_crs=TARGET_CRS, resampling=Resampling.nearest,
)
extent = grid_to_extent(cs_transform, cs_width, cs_height)

selected = LAYERS_TO_PLOT or list(layers.keys())

for layer_name in selected:
    if layer_name not in layers:
        print(f"⚠️  Unknown layer: {layer_name}")
        continue
    cfg = layers[layer_name]
    files = sorted(glob.glob(cfg["path"]))
    if not files:
        print(f"⚠️  No files for {layer_name}: {cfg['path']}")
        continue

    for fp in files:
        # Upsample climate to the CS grid
        clim_aligned = reproject_to_grid(
            fp, cs_transform, cs_width, cs_height,
            dst_crs=TARGET_CRS, resampling=Resampling.bilinear,
        )

        # Classify
        (cs_valid, clim_valid), mask_2d = mask_valid_pixels(cs_aligned, clim_aligned)
        x_breaks, y_breaks, codes = bivariate_classify(
            cs_valid, clim_valid, n_classes=N_CLASSES, method="quantile"
        )
        codes_2d = unflatten_to_raster(codes, mask_2d, fill=-1)

        # Save everything
        run_dir = OUT_ROOT / f"{layer_name}__{Path(fp).stem}"
        save_bivariate_outputs(
            run_dir,
            bivar_map=codes_2d,
            extent=extent,
            color_dict=layers[layer_name].get("bivar"),
            x_label="Conservation Score →",
            y_label=f"{cfg.get('label', layer_name)} →",
            n_classes=N_CLASSES,
            x_breaks=x_breaks,
            y_breaks=y_breaks,
            meta_extra={
                "layer_name": layer_name,
                "climate_file": str(fp),
                "cs_file": str(CS_PATH),
                "x_variable": "Conservation Score (CS)",
                "y_variable": cfg.get("label", layer_name),
                "y_units": cfg.get("cbar_label", ""),
                "classification_method": "quantile",
            },
        )
        print(f"✓ saved {run_dir.name}/")